# Legal Information Extraction Pipeline (4E Framework)

Extracts structured legal information from Indian court judgments using a Hugging Face instruction-tuned LLM. No annotations are used — every field is inferred directly from `judgment_text`.

**Output schema per judgment**
```json
{ "case_type", "subject", "object", "objective_aspect", "subjective_aspect", "reasoning" }
```

**Pipeline:** install → imports → config → model load → data load → prompt → inference (with JSON parsing, retries, error handling, perf monitoring) → save to `indilex_extracted_output.xlsx`.

> Switch models by changing `MODEL_NAME` in the config cell. Llama / Mistral / Gemma / DeepSeek instruct models work unchanged because everything routes through the tokenizer chat template.

## 1. Package Installation

In [1]:
# Run once, then RESTART THE KERNEL before continuing.
# Qwen2.5 needs transformers >= 4.37. This pin works on Python 3.8/3.9
# and avoids the 4.46.0-4.46.2 regression. On Python >= 3.10 you may
# instead use a newer release: %pip install -q -U transformers
%pip install -q -U "transformers==4.46.3" "tokenizers>=0.20,<0.21" \
    "accelerate>=0.26" "huggingface_hub>=0.24" sentencepiece pandas openpyxl tqdm

    torch (>=1.7.*)
           ~~~~~~^
Note: you may need to restart the kernel to use updated packages.


### (Optional) Free GPU memory without leaving Jupyter
If you hit *CUDA out of memory*, the usual cause is leftover python processes from earlier sessions holding the GPU. Run the cell below to see them, then kill stray PIDs from a terminal with `kill -9 <PID>`. Killing from inside the notebook is unsafe (you might kill this kernel), so only inspect here.

In [4]:
# Inspect GPU usage (read-only). Kill stray PIDs from a TERMINAL, not here.
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Tue Jun 23 17:40:05 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090         On | 00000000:3B:00.0 Off |                  N/A |
|  0%   52C    P8               26W / 350W|      8MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
!kill 34547 40344

/bin/bash: line 0: kill: (34547) - No such process


In [2]:
import transformers
print("transformers:", transformers.__version__)
from transformers import Qwen2ForCausalLM
print("Qwen2 import OK")

transformers: 4.46.3
Qwen2 import OK


## 2. Imports

In [3]:
import os
import re
import json
import time
import traceback

import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

import transformers
from packaging import version
print("transformers:", transformers.__version__)
assert version.parse(transformers.__version__) >= version.parse("4.37"), (
    "Qwen2.5 needs transformers >= 4.37. Run the install cell above, then RESTART the kernel."
)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

transformers: 4.46.3
torch: 2.6.0+cu124
CUDA available: True


/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 3. Configuration

Single source of truth. Change `MODEL_NAME` to swap models — no other edits needed.

In [15]:
# ----------------------------- MODEL -----------------------------
# Default. Drop-in alternatives (uncomment one):
 MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

 #MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# MODEL_NAME = "CohereForAI/aya-expanse-8b"

# MODEL_NAME = "google/gemma-2-9b-it"

# MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"

# ----------------------------- DATA ------------------------------
INPUT_PATH   = "Allahabad_criminal_merged.xlsx"   # must contain judgment_text (+ ideally case_id, language)
OUTPUT_PATH  = "indilex_extracted_output_Qwen.xlsx"
TEXT_COLUMN  = "judgment_text"

# ----------------------------- RUN MODE --------------------------
SAMPLE_MODE  = True   # True -> first SAMPLE_SIZE rows; False -> full dataset (no other code changes)
SAMPLE_SIZE  = 5

# ----------------------------- GENERATION ------------------------
MAX_NEW_TOKENS   = 700
TEMPERATURE      = 0.2
TOP_P            = 0.9
DO_SAMPLE        = True
MAX_INPUT_TOKENS = 7000   # truncate very long judgments from the left to fit context
MAX_RETRIES      = 3      # extra attempts when JSON parsing fails

# ----------------------------- 4E SCHEMA -------------------------
SCHEMA_KEYS = ["case_type", "subject", "object",
               "objective_aspect", "subjective_aspect", "reasoning"]
CASE_TYPES  = ["Criminal", "Civil", "Constitutional", "Administrative"]

## 4. Model Loading

In [16]:
import torch, gc

# --- Free any leftover GPU memory from previous loads in THIS kernel ---
for _v in ("model", "tokenizer"):
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()

# --- Show GPU memory before loading so OOM is caught early ---
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f"GPUs visible: {n}")
    for d in range(n):
        free, total = torch.cuda.mem_get_info(d)
        print(f"  cuda:{d}  free {free/1e9:5.1f} GB / total {total/1e9:5.1f} GB")
    # Qwen2.5-7B fp16 needs ~16 GB. Need a single GPU with enough room.
    need_gb = 16
    pick = None
    for d in range(n):
        free, _ = torch.cuda.mem_get_info(d)
        if free/1e9 >= need_gb:
            pick = d
            break
    if pick is None:
        raise RuntimeError(
            f"No single GPU has ~{need_gb} GB free for 7B fp16. "
            "Free GPU memory first: in a terminal run `nvidia-smi`, find your own "
            "python PIDs, and `kill -9 <PID>` the ones holding memory. Then re-run this cell."
        )
    print(f"Loading on cuda:{pick}")
else:
    pick = None
    print("WARNING: CUDA not available -> will load on CPU (very slow).")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

try:
    if pick is not None:
        # Pin to ONE chosen GPU so the model does not get split across both cards
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        ).to(f"cuda:{pick}")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
except torch.cuda.OutOfMemoryError as e:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "CUDA out of memory while loading. The GPU filled up between the check and the load "
        "(another process grabbed it). Run `nvidia-smi` in a terminal, kill stray python PIDs, "
        "then RESTART THE KERNEL and run from the top."
    ) from e
except (ValueError, KeyError) as e:
    raise RuntimeError(
        f"Failed to load {MODEL_NAME}. Likely transformers too old. You have "
        f"{__import__('transformers').__version__}; Qwen2.5 needs >= 4.37."
    ) from e

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded on:", next(model.parameters()).device)

GPUs visible: 2
  cuda:0  free  24.3 GB / total  25.4 GB
  cuda:1  free  24.3 GB / total  25.4 GB
Loading on cuda:0


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [17]:
# NOTE: This cell is intentionally disabled.
# The model is already loaded on a single GPU in the model-loading cell above.
# Do NOT run a second .to("cuda") load here -- that caused the earlier
# "out of memory" and "model is not defined" errors.
print("Skip this cell. Model is loaded above on:", next(model.parameters()).device)

Skip this cell. Model is loaded above on: cuda:0


## 5. Dataset Loading

Loads the Excel file, validates the text column, and synthesizes `case_id` / `language` if missing so the pipeline never hard-fails on schema drift.

In [18]:
df = pd.read_excel(INPUT_PATH)

if TEXT_COLUMN not in df.columns:
    raise ValueError(f"'{TEXT_COLUMN}' not found. Available columns: {list(df.columns)}")

if "case_id" not in df.columns:
    df["case_id"] = df.index.astype(str)
if "language" not in df.columns:
    df["language"] = "unknown"

print("Full dataset shape:", df.shape)

work_df = (df.head(SAMPLE_SIZE).copy() if SAMPLE_MODE else df.copy())
print(f"Mode: {'SAMPLE' if SAMPLE_MODE else 'FULL'}  ->  processing {len(work_df)} records")
work_df.head()

Full dataset shape: (199, 10)
Mode: SAMPLE  ->  processing 5 records


,case_id,language,case_type,judgment_text,subject,object,objective_aspect,subjective_aspect,reasoning,legal provision
0,ALL_CRIM_000001,Hindi,Criminal,Court No.=27\n\nAPPLICATION UIS 482 No. - 741 ...,Dinesh Yadav @ Dinesh Kumar And 7 Others,Wife/Complainant,Assault; Bigamy; Cruelty; Intentional Insult; ...,Dowry Demand,NaN,NaN
1,ALL_CRIM_000002,Hindi,Criminal,‘Neutral Citation No. - 2023:AHC:145371\n\nCou...,Kamlesh Jaiswal Alias Monu Jaiswal And Another,State/Society,Criminal Law Amendment Act Offence,Not Clearly Specified,NaN,NaN
2,ALL_CRIM_000003,Hindi,Criminal,‘Neutral Citation No. - 2024:AHC-LKO:1629\nCou...,Bahal Khan And 4 Others,Complainant/Informant,Rioting; Rioting With Deadly Weapon; Assault; ...,Intent/Knowledge To Cause Death Or Serious Harm,NaN,NaN
3,ALL_CRIM_000004,Hindi,Criminal,"Court No, -27\n(Case := APPLICATION U/S 482 No...",Banshi Lal,Complainant/Informant,Kidnapping Or Abduction,Intent To Compel Wrongful Restraint/Confinement,NaN,NaN
4,ALL_CRIM_000005,Hindi,Criminal,Neutral Citation No. - 2024:AHC-LKO:7688\nCour...,Mohd. Mahfooz,Wife/Complainant,Assault; Mischief; Cruelty; Intentional Insult,Dowry Demand,NaN,NaN


## 6. Prompt Construction

A system prompt fixes the role and the strict JSON contract; the user prompt carries the 4E definitions and the judgment. The judgment is the only data passed to the model.

In [19]:
SYSTEM_PROMPT = (
    "You are an expert Indian legal analyst. You read court judgments and extract "
    "structured legal information. You return ONLY a single valid JSON object. "
    "You never output markdown, code fences, or any text outside the JSON."
)

def build_user_prompt(judgment_text: str) -> str:
    return f"""Read the court judgment carefully and extract structured legal information. Infer every field directly from the judgment text. Do not hallucinate facts or fabricate findings.

Return a JSON object with exactly these keys: case_type, subject, object, objective_aspect, subjective_aspect, reasoning.

FIELD DEFINITIONS
- case_type: one of {CASE_TYPES}.
- subject: the primary legal actor. (Criminal -> Accused, Civil -> Defendant, Constitutional -> State Respondent, Administrative -> Administrative Authority.)
- object: the affected party. (Criminal -> Victim, Civil -> Plaintiff, Constitutional -> Petitioner, Administrative -> Aggrieved Party.)
- objective_aspect: the observable legal act, dispute, allegation, violation, challenged action, cause of action, or legally relevant event.
- subjective_aspect: the intent, motive, legal claim, remedy sought, writ relief, legal objective, or grounds for review.
- reasoning: 3-5 sentences explaining the court's ACTUAL reasoning and the final outcome, grounded in the judgment. Focus on judicial findings. Do NOT merely repeat FIR allegations, pleadings, or arguments.

RULES
- If a field cannot be confidently determined, use the string "Unknown".
- Output ONLY the JSON object. No markdown. No code blocks. No commentary.

JUDGMENT:
'''{judgment_text}'''"""

## 7. Inference

Builds the chat-templated input, truncates over-long judgments, generates, and returns the decoded completion plus token/timing stats for performance monitoring.

In [20]:
def generate(judgment_text: str):
    """Run one generation. Returns (response_text, n_new_tokens, elapsed_seconds)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_prompt(str(judgment_text))},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=MAX_INPUT_TOKENS,
    ).to(model.device)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - t0

    gen_ids = outputs[0][inputs.input_ids.shape[1]:]
    n_new_tokens = int(gen_ids.shape[0])
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return response, n_new_tokens, elapsed

## 8. JSON Parsing

Strips stray code fences, extracts the first balanced `{...}` block, parses it, and normalizes the result to the fixed schema (missing keys become `"Unknown"`, case_type is snapped to the allowed set).

In [21]:
def _extract_json_block(raw: str):
    """Return the first balanced {...} substring, or None."""
    s = raw.strip()
    s = re.sub(r"^```(?:json)?", "", s).strip()
    s = re.sub(r"```$", "", s).strip()
    start = s.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(s)):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return s[start:i + 1]
    return None


def parse_json(raw: str):
    """Parse model output into the fixed schema. Returns dict or None on failure."""
    block = _extract_json_block(raw)
    if block is None:
        return None
    try:
        data = json.loads(block)
    except json.JSONDecodeError:
        return None
    if not isinstance(data, dict):
        return None

    out = {}
    for k in SCHEMA_KEYS:
        v = data.get(k, "Unknown")
        if v is None or (isinstance(v, str) and not v.strip()):
            v = "Unknown"
        out[k] = v if isinstance(v, str) else json.dumps(v, ensure_ascii=False)

    # Normalize case_type to the allowed vocabulary when possible
    ct = out["case_type"].strip().lower()
    out["case_type"] = next((c for c in CASE_TYPES if c.lower() in ct), out["case_type"])
    return out

## 9. Error Handling, Retries & Per-Record Extraction

`extract_one` wraps generation + parsing with retries on malformed JSON, keeps the raw output, and never raises — a failed record is logged and the pipeline continues.

In [22]:
def extract_one(judgment_text: str):
    """Robust single-record extraction.

    Returns a dict with the 4E fields plus bookkeeping:
    raw_output, parse_ok, attempts, error, n_tokens, gen_time.
    """
    last_raw, last_err = "", None
    total_tokens, total_time = 0, 0.0

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw, n_tok, elapsed = generate(judgment_text)
            last_raw = raw
            total_tokens += n_tok
            total_time += elapsed

            parsed = parse_json(raw)
            if parsed is not None:
                parsed.update({
                    "raw_output": raw, "parse_ok": True, "attempts": attempt,
                    "error": "", "n_tokens": total_tokens,
                    "gen_time": round(total_time, 3),
                })
                return parsed
            last_err = "JSON parse failed"
        except Exception as e:  # OOM, generation errors, etc. — keep going
            last_err = f"{type(e).__name__}: {e}"
            last_raw = last_raw or traceback.format_exc()

    # All attempts failed -> Unknown-filled record, still schema-valid
    failed = {k: "Unknown" for k in SCHEMA_KEYS}
    failed.update({
        "raw_output": last_raw, "parse_ok": False, "attempts": MAX_RETRIES,
        "error": last_err or "unknown error",
        "n_tokens": total_tokens, "gen_time": round(total_time, 3),
    })
    return failed

### Quick check on one record

In [23]:
_demo = extract_one(work_df.iloc[0][TEXT_COLUMN])
print("parse_ok:", _demo["parse_ok"], "| attempts:", _demo["attempts"])
print(json.dumps({k: _demo[k] for k in SCHEMA_KEYS}, indent=2, ensure_ascii=False))

parse_ok: True | attempts: 1
{
  "case_type": "Criminal",
  "subject": "Prathis 1 and 2 to 8",
  "object": "State of UP through Principal Secretary, Home Department, Lucknow",
  "objective_aspect": "Charges under sections 323, 494, 498A, 504, 506 of the Indian Penal Code based on FIR dated 16.6.2020",
  "subjective_aspect": "Agreement between parties to withdraw the criminal proceedings",
  "reasoning": "The court accepted the application as a settlement agreement was reached between the parties. The criminal charges against Prathis 1 and 2 to 8 under sections 323, 494, 498A, 504, 506 of the Indian Penal Code, based on the FIR dated 16.6.2020, were withdrawn."
}


## 10. Batch Inference — Progress Tracking & Performance Monitoring

Iterates over all selected records with a `tqdm` progress bar, printing per-judgment time and tokens/sec and accumulating totals.

In [24]:
records, failed_log = [], []
run_start = time.time()

pbar = tqdm(work_df.iterrows(), total=len(work_df), desc="Extracting")
for idx, row in pbar:
    res = extract_one(row[TEXT_COLUMN])
    res["case_id"]  = row.get("case_id", idx)
    res["language"] = row.get("language", "unknown")
    records.append(res)

    if not res["parse_ok"]:
        failed_log.append({"case_id": res["case_id"], "error": res["error"]})

    tps = (res["n_tokens"] / res["gen_time"]) if res["gen_time"] > 0 else 0.0
    pbar.set_postfix({
        "ok": res["parse_ok"],
        "t/judg": f"{res['gen_time']:.1f}s",
        "tok": res["n_tokens"],
        "tok/s": f"{tps:.1f}",
    })

total_runtime = time.time() - run_start
n_ok   = sum(r["parse_ok"] for r in records)
sum_tok = sum(r["n_tokens"] for r in records)

print("\n================ RUN SUMMARY ================")
print(f"Records processed : {len(records)}")
print(f"Parsed OK         : {n_ok}")
print(f"Failed            : {len(records) - n_ok}")
print(f"Total tokens      : {sum_tok}")
print(f"Total runtime     : {total_runtime:.1f}s")
if records:
    print(f"Avg time/judgment : {total_runtime / len(records):.1f}s")
if total_runtime > 0:
    print(f"Overall tokens/sec: {sum_tok / total_runtime:.1f}")
if failed_log:
    print("Failed records    :", failed_log)

Extracting:   0%|          | 0/5 [00:00<?, ?it/s]


================ RUN SUMMARY ================
Records processed : 5
Parsed OK         : 5
Failed            : 0
Total tokens      : 1013
Total runtime     : 32.9s
Avg time/judgment : 6.6s
Overall tokens/sec: 30.8


In [25]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.parameters()).device)

CUDA available: True
Model device: cuda:0


## 11. Result Saving

Appends the six 4E fields (plus `raw_output` for auditability) to the original rows and writes `indilex_extracted_output.xlsx`. Merge is by `case_id` so it works in both sample and full mode.

In [26]:
results_df = pd.DataFrame(records)

# Columns to append to the original dataset
append_cols = SCHEMA_KEYS + ["raw_output", "parse_ok", "attempts", "error"]
results_slim = results_df[["case_id"] + append_cols].copy()

base = (work_df if SAMPLE_MODE else df).copy()
base["case_id"] = base["case_id"].astype(str)
results_slim["case_id"] = results_slim["case_id"].astype(str)

# Drop any pre-existing 4E columns before merging to avoid suffix clashes
base = base.drop(columns=[c for c in append_cols if c in base.columns], errors="ignore")

final_df = base.merge(results_slim, on="case_id", how="left")
final_df.to_excel(OUTPUT_PATH, index=False)
print(f"Saved {len(final_df)} rows -> {OUTPUT_PATH}")

final_df[["case_id"] + SCHEMA_KEYS].head(10)

Saved 5 rows -> indilex_extracted_output_Qwen.xlsx


/tmp/ipykernel_20852/3865926114.py:15: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  final_df.to_excel(OUTPUT_PATH, index=False)


,case_id,case_type,subject,object,objective_aspect,subjective_aspect,reasoning
0,ALL_CRIM_000001,Criminal,Prathaigana (Applicants),"State of UP through Principal Secretary, Home ...","Challenging the arrest under sections 323, 494...",The applicants seek to have the criminal proce...,The court accepted the plea application as the...
1,ALL_CRIM_000002,Criminal,Kamlesh Jaiswal Alias Monu Jaiswal And Another,State of U.P.,Application under Section 438 Cr.P.C. for anti...,The applicant claims to be innocent and has no...,The court considered the arguments presented b...
2,ALL_CRIM_000003,Criminal,Accused,State of U.P.,Appeal under Section 482 to quash criminal cha...,The accused parties seek to have the criminal ...,The court agreed to quash the criminal charges...
3,ALL_CRIM_000004,Criminal,Bansi Lal,State of UP,Appeal against the final order and sentence da...,The applicant seeks to challenge the validity ...,The court found no valid challenge to the fina...
4,ALL_CRIM_000005,Criminal,Prisoner,State of UP through Principal Secretary Home D...,Appeal under Section 482 of CrPC to quash crim...,The applicant and the opposite party have reac...,The court accepted the application under Secti...


## 12. Run the Full Dataset

To process everything, set `SAMPLE_MODE = False` in the **Configuration** cell, then re-run from **Dataset Loading** downward. No other code changes are required.